In [1]:
import pandas as pd
import numpy as np

from xbbg import blp
import pdblp

import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

import matplotlib.pyplot as plt
from pandas.plotting import table


from xbbg import blp
from datetime import datetime, timedelta
from pandas.tseries.offsets import DateOffset

from scipy.stats import percentileofscore

from typing import List, Dict, Optional, Union

In [2]:

class EconomicDataManager:
    """
    A class to manage economic indicator data retrieval and processing.
    """
    # Class-level constants for Bloomberg fields
    DEFAULT_FIELDS = [
        "NAME",
        "TIME", 
        "ECO_RELEASE_TIME",
        "ECO_RELEASE_DT",
        "ACTUAL_RELEASE",
        "BN_SURVEY_MEDIAN",
        "BN_SURVEY_AVERAGE", 
        "BN_SURVEY_HIGH",
        "BN_SURVEY_LOW",
        "FORECAST_STANDARD_DEVIATION",
        "BN_SURVEY_NUMBER_OBSERVATIONS"]
    
    FIELD_RENAME_MAP = {
        "ECO_RELEASE_DT": "ReleaseDate",
        "ACTUAL_RELEASE": "Actual", 
        "BN_SURVEY_MEDIAN": "SMed",
        "BN_SURVEY_AVERAGE": "SAve",
        "BN_SURVEY_HIGH": "SHigh",
        "BN_SURVEY_LOW": "SLow",
        "FORECAST_STANDARD_DEVIATION": "ForecastSDTv",
        "BN_SURVEY_NUMBER_OBSERVATIONS": "NumbSurvey"}
    # -------------------------------------------------------------------------------------------
    # INITIALIZING Functions
    # Initialize the EconomicDataManager - Input: blp
    def __init__(self, bloomberg_api, indicators_dict: Optional[Dict] = None):
        self.blp = bloomberg_api
        self.indicators = indicators_dict or {}
    # Add indicators for a specific currency/region - Input: CCY Code, Indicator List
    def add_indicators(self, currency_code: str, indicators: List[Dict]) -> None:
        self.indicators[currency_code] = indicators
    # -------------------------------------------------------------------------------------------
    # CORE DATA Retriving Function
    # Retrieve past and future data for a specific ticker - Input: Ticker, Days
    def get_past_future_data(self, ticker: str, days: int = 100, 
                           custom_fields: Optional[List[str]] = None) -> pd.DataFrame:
        fields = custom_fields or self.DEFAULT_FIELDS
        start_date = (datetime.today() - timedelta(days=days)).strftime('%Y-%m-%d')
        end_date = (datetime.today() + timedelta(days=days)).strftime('%Y-%m-%d')
        try:
            data = self.blp.bdh(
                tickers=ticker,
                flds=fields,
                start_date=start_date,
                end_date=end_date)
            df = data.rename(columns=self.FIELD_RENAME_MAP)
            df.columns = df.columns.droplevel(0)
            df['ReleaseDate'] = pd.to_datetime(
                df['ReleaseDate'].astype(str).str.split('.').str[0],
                format='%Y%m%d',
                errors='coerce')
            df = df.dropna(subset=['ReleaseDate'])
            df = df.sort_values('ReleaseDate')
            return df
        except Exception as e:
            return pd.DataFrame()  # Return empty DataFrame on error
    # Get only past data releases for a ticker - Input: Ticker, Days
    def get_past_data(self, ticker: str, days: int = 100) -> pd.DataFrame:
        df = self.get_past_future_data(ticker, days)
        today = pd.Timestamp.today().normalize()
        return df[df['ReleaseDate'] < today]
    # Get only future data releases for a ticker - Input: Ticker, Days
    def get_future_data(self, ticker: str, days: int = 100) -> pd.DataFrame:
        df = self.get_past_future_data(ticker, days)
        today = pd.Timestamp.today().normalize()
        return df[df['ReleaseDate'] >= today]
    # -------------------------------------------------------------------------------------------
    # DATE OF EVENT Functions
    # Get the next release date for a specific ticker - Input: Ticker, Days
    def get_next_release_date(self, ticker: str, days: int = 100) -> Optional[str]:
        try:
            future_df = self.get_future_data(ticker, days)
            if future_df.empty:
                return None
            return str(future_df.iloc[0]['ReleaseDate'].date())
        except Exception as e:
            return None
  
    # -------------------------------------------------------------------------------------------
    # SCHEDULE OF EVENTS Functions
    # Get upcoming release schedule for multiple indicators - input: Indicators, days
    def get_release_schedule(self, indicators: List[Dict], days: int = 100) -> pd.DataFrame:
        results = []
        for indicator in indicators:
            try:
                release_date = self.get_next_release_date(indicator["Ticker"], days)
                if release_date:  # Only add if we got a valid date
                    results.append({
                        "Country": indicator["Country"],
                        "Data": indicator["Data"], 
                        "Ticker": indicator["Ticker"],
                        "Release Date": release_date})
            except Exception as e:
                continue
        df = pd.DataFrame(results)
        df["Release Date"] = pd.to_datetime(df["Release Date"], errors='coerce')
        df = df.dropna(subset=["Release Date"])
        df = df.sort_values(by="Release Date")
        df.insert(
            loc=df.columns.get_loc("Release Date") + 1,
            column="Weekday", 
            value=df["Release Date"].dt.day_name().str[:3])
        df = self._add_week_grouping(df)
        return df
   # -------------------------------------------------------------------------------------------
    # UTILITY Functions
    # Filter DataFrame by date range - Input: df w/ ['Release Date'], Start Date, End Date
    def filter_by_date_range(self, df: pd.DataFrame, start_date: str, end_date: str) -> pd.DataFrame:
        start = pd.to_datetime(start_date)
        end = pd.to_datetime(end_date)
        return df[(df['Release Date'] >= start) & (df['Release Date'] <= end)]
    # Add week grouping to DataFrame based on release dates
    def _add_week_grouping(self, df: pd.DataFrame) -> pd.DataFrame:
        today = pd.Timestamp.today().normalize()
        start_of_this_week = today - pd.Timedelta(days=today.weekday())
        df["Week"] = ((df["Release Date"] - start_of_this_week).dt.days // 7).clip(lower=0)
        return df
    

    # -------------------------------------------------------------------------------------------

    # Get release schedule for a specific region
    def get_currency_schedule(self, currency_code: str, days: int = 100) -> pd.DataFrame:
        if currency_code not in self.indicators:
            raise ValueError(f"Currency code '{currency_code}' not found in indicators")
        return self.get_release_schedule(self.indicators[currency_code], days)
    
    
    
    # Get combined release schedule for all loaded currencies - Input: Days
    def get_all_currencies_schedule(self, days: int = 100) -> pd.DataFrame:
        all_indicators = []
        for currency_indicators in self.indicators.values():
            all_indicators.extend(currency_indicators)
        return self.get_release_schedule(all_indicators, days)




# Factory function to set up EconomicDataManager with predefined indicators
def setup_economic_data_manager(bloomberg_api):
    # Define your indicator dictionaries here (from your original data)
    USD_indicators = [
        {"Country": "US", "Data": "CPI YoY", "Ticker": "CPI YOY Index"},
        {"Country": "US", "Data": "CPI MoM", "Ticker": "CPI CHNG Index"},
        {"Country": "US", "Data": "Core CPI YoY", "Ticker": "CPURNSA Index"},
        {"Country": "US", "Data": "Core CPI MoM", "Ticker": "CPUPXCHG Index"},

        {"Country": "US", "Data": "PPI MoM", "Ticker": "FDIDFDMO Index"},
        {"Country": "US", "Data": "PPI YoY", "Ticker": "FDIUFDYO Index"},

        {"Country": "US", "Data": "Unemployment Rate", "Ticker": "USURTOT Index"},
        {"Country": "US", "Data": "Nonfarm Payrolls (NFPs)", "Ticker": "NFP TCH Index"},
        {"Country": "US", "Data": "Initial Jobless Claims", "Ticker": "INJCJC Index"},
        {"Country": "US", "Data": "Continuing Claims", "Ticker": "INJCSP   Index"},

        {"Country": "US", "Data": "Fed Rate Decision", "Ticker": "FDTR Index"},

        {"Country": "US", "Data": "Retail Sales MoM", "Ticker": "RSTAMOM Index"},
        {"Country": "US", "Data": "Retail Sales Ex Auto MoM", "Ticker": "RSTAXMOM Index"},
        {"Country": "US", "Data": "GDP QoQ Annualized", "Ticker": "GDP CQOQ Index"},

        {"Country": "US", "Data": "Core PCE YoY", "Ticker": "PCE CYOY Index"},
        {"Country": "US", "Data": "Core PCE MoM", "Ticker": "PCE CMOM Index"},
        {"Country": "US", "Data": "University of Michigan Sentiment", "Ticker": "CONSSENT Index"},

        {"Country": "US", "Data": "ISM Manufacturing", "Ticker": "NAPMPMI Index"},
        {"Country": "US", "Data": "ISM Services", "Ticker": "NAPMNMI Index"},
        {"Country": "US", "Data": "Industrial Production MoM", "Ticker": "IPMGCHNG Index"},
        {"Country": "US", "Data": "Durable Goods Orders MoM", "Ticker": "DGNOCHNG Index"},
        {"Country": "US", "Data": "Housing Starts", "Ticker": "NHCHST Index"},
        {"Country": "US", "Data": "Building Permits", "Ticker": "NHSPATOT Index"},
        ]

    EUR_indicators = [
        {"Country": "EU", "Data": "CPI YoY", "Ticker": "ECCPEMUY Index"},
        {"Country": "EU", "Data": "Core CPI YoY", "Ticker": "CPEXEMUY Index"},
        {"Country": "EU", "Data": "CPI MoM", "Ticker": "ECCPEMUM Index"},
        
        {"Country": "EU", "Data": "GDP QoQ", "Ticker": "ECCPEMUM Index"},
        {"Country": "EU", "Data": "GDP YoY", "Ticker": "EUGNEMUY Index"},

        {"Country": "EU", "Data": "PPI MoM", "Ticker": "EUPPEMUM Index"},
        {"Country": "EU", "Data": "PPI Finished Goods", "Ticker": "EUPPEMUY Index"},
        
        {"Country": "EU", "Data": "Unemployment Rate", "Ticker": "UMRTEMU Index"},
        {"Country": "EU", "Data": "Employment YoY", "Ticker": "EMEMULYY Index"},
        {"Country": "EU", "Data": "Employment QoQ", "Ticker": "EMEMULQQ Index"},
        
        {"Country": "EU", "Data": "ECB Rate Decision", "Ticker": "EURR002W Index"},
        
        {"Country": "Germany", "Data": "IFO Gern Business Climate", "Ticker": "GRIFPBUS Index"}, 
        {"Country": "Germany", "Data": "ZEW Germ Eco Growth Expectations", "Ticker": "GRZEWI Index"}, 
        {"Country": "EU", "Data": "Consumer Confidence", "Ticker": "EUCCEMU Index"},
        {"Country": "EU", "Data": "Economic Confidence", "Ticker": "EUESEMU  Index"},
        
        {"Country": "EU", "Data": "Manufacturing PMI", "Ticker": "MPMIEZMA Index"},
        {"Country": "EU", "Data": "Services PMI", "Ticker": "MPMIEZSA Index"},
        {"Country": "EU", "Data": "Composite PMI", "Ticker": "MPMIEZCA Index"},
        
        {"Country": "EU", "Data": "Industrial Production MoM", "Ticker": "EUITEMUM Index"},
        {"Country": "EU", "Data": "Retail Sales YoY", "Ticker": "RSWAEMUY Index"},
        
        {"Country": "EU", "Data": "Trade Balance", "Ticker": "XTTBEZ Index"}
        ]

    JPY_indicators = [
        {"Country": "Japan", "Data": "CPI YoY", "Ticker": "JNCPIYOY Index"},
        {"Country": "Japan", "Data": "CPI Ex Fresh Food YoY (Core)", "Ticker": "JNCPIXFF Index"},
        {"Country": "Japan", "Data": "CPI Ex Fresh Food & Energy YoY (Core-Core)", "Ticker": "JCPTEFFE Index"},
        {"Country": "Japan", "Data": "PPI YoY", "Ticker": "JNWSDYOY Index"},

        {"Country": "Japan", "Data": "Unemployment Rate", "Ticker": "JNUE Index"},
        {"Country": "Japan", "Data": "Job-To-Applicant Ratio", "Ticker": "JBTARATE Index"},

        {"Country": "Japan", "Data": "Industrial Production YoY", "Ticker": "JNIPYOY Index"},
        {"Country": "Japan", "Data": "Tankan Large mfg Index", "Ticker": "JNTSMFG Index"},
        {"Country": "Japan", "Data": "Tankan Large mfg Outlook", "Ticker": "JPTFLMFG Index"},
        {"Country": "Japan", "Data": "S&P JPN PMI", "Ticker": "JPTFLMFG Index"},

        {"Country": "Japan", "Data": "Retail Sales MoM", "Ticker": "JNRETMOM Index"},

        {"Country": "Japan", "Data": "Trade Balance", "Ticker": "JNTBAL Index"},
        {"Country": "Japan", "Data": "Exports YoY", "Ticker": "JNTBEXPY Index"},
        {"Country": "Japan", "Data": "Imports YoY", "Ticker": "JNTBIMPY Index"},

        {"Country": "Japan", "Data": "BOJ Rate Decision", "Ticker": "BOJDTR Index"},
        {"Country": "Japan", "Data": "GDP Annualized QoQ", "Ticker": "JGDPQGDP Index"},  
        ]

    GBP_indicators = [
        {"Country": "UK", "Data": "GDP QOQ", "Ticker": "UKGRABIQ Index"},

        {"Country": "UK", "Data": "CPI YoY", "Ticker": "UKRPCJYR Index"},
        {"Country": "UK", "Data": "CPI MoM", "Ticker": "UKRPCJMR Index"},
        {"Country": "UK", "Data": "Core CPI YoY", "Ticker": "UKHCA9IQ Index"},
        {"Country": "UK", "Data": "RPI YoY", "Ticker": "UKRPYOY  Index"},

        {"Country": "UK", "Data": "Unemployment Claims", "Ticker": "UKUEMOM Index"},

        {"Country": "UK", "Data": "Retail Sales MoM", "Ticker": "UKUEMOM Index"},
        {"Country": "UK", "Data": "Industrial Production YoY", "Ticker": "UKMPIYOY Index"},
        {"Country": "UK", "Data": "Gfk Consumer Confidence", "Ticker": "UKCCI Index"},

        {"Country": "UK", "Data": "Manufacturing PMI", "Ticker": "UKCCI Index"},
        {"Country": "UK", "Data": "Service PMI", "Ticker": "MPMIGBSA Index"},
        {"Country": "UK", "Data": "Composite PMI", "Ticker": "MPMIGBCA Index"},

        {"Country": "UK", "Data": "Trade Balance", "Ticker": "MPMIGBCA Index"},

        {"Country": "UK", "Data": "BoE Rate Decision", "Ticker": "UKBRBASE Index"},
    ]

    AUD_indicators = [
        {"Country": "AU", "Data": "CPI YoY", "Ticker": "AUCPIYOY Index"},
        {"Country": "AU", "Data": "CPI QoQ", "Ticker": "AUCPICHG Index"},
        {"Country": "AU", "Data": "Trimmed Mean CPI YoY", "Ticker": "RBCPTRIY Index"},
        {"Country": "AU", "Data": "Trimmed Mean CPI QoQ", "Ticker": "RBCPTRIQ Index"},
        {"Country": "AU", "Data": "Melbrn Infl Gauge QoQ", "Ticker": "TDMIMOM Index"},
        {"Country": "AU", "Data": "Melbrn Cnsmr Confi MoM", "Ticker": "TDMIMOM Index"},
        
        {"Country": "AU", "Data": "Unemployment Rate", "Ticker": "AULFUNEM Index"},
        {"Country": "AU", "Data": "Employment Change", "Ticker": "AULFEMPC Index"},
        {"Country": "AU", "Data": "Participation Rate", "Ticker": "AULFPART Index"},
        
        {"Country": "AU", "Data": "Industrial Production YoY", "Ticker": "AUIPTYOY Index"},  # AUD PMI
        
        {"Country": "AU", "Data": "Retail Sales MoM", "Ticker": "AURSTSA Index"},
        
        {"Country": "AU", "Data": "Trade Balance", "Ticker": "AUITGSB Index"},
        {"Country": "AU", "Data": "RBA CAsh Rate", "Ticker": "RBATCTR  Index"},

        {"Country": "AU", "Data": "RBA Cash Rate Target", "Ticker": "RBATCTR Index"},
        {"Country": "AU", "Data": "GDP QoQ", "Ticker": "AUNAGDPC Index"},
        {"Country": "AU", "Data": "GDP YoY", "Ticker": "AUNAGDPY Index"},
    ]

    NZD_indicators = [
        {"Country": "NZ", "Data": "CPI YoY", "Ticker": "NZCPICHG Index"},
        {"Country": "NZ", "Data": "CPI QoQ", "Ticker": "NZCPIYOY Index"},

        {"Country": "NZ", "Data": "Unemployment Rate", "Ticker": "NZLFUNER Index"},
        {"Country": "NZ", "Data": "Employment Change QoQ", "Ticker": "NZLFQOQ  Index"},

        {"Country": "NZ", "Data": "GDP QoQ", "Ticker": "NZNTGDPC Index"},
        {"Country": "NZ", "Data": "GDP YoY", "Ticker": "NZNTGDPY Index"},

        {"Country": "NZ", "Data": "Retail Sales Ex Inflation QoQ", "Ticker": "NZRREXIN Index"},

        {"Country": "NZ", "Data": "Trade Balance", "Ticker": "NZMTBAL Index"},
        {"Country": "NZ", "Data": "Exports", "Ticker": "NZMTEXP Index"},
        {"Country": "NZ", "Data": "Imports", "Ticker": "NZMTIMP Index"},

        {"Country": "NZ", "Data": "RBNZ Official Cash Rate", "Ticker": "NZOCR Index"},

        {"Country": "NZ", "Data": "Business Mnft PMI", "Ticker": "NZPMISA Index"},
        {"Country": "NZ", "Data": "Consumer Confidence", "Ticker": "NZANCCT Index"},
    ]

    CHF_indicators = [
        {"Country": "CHF", "Data": "CPI YoY", "Ticker": "SZCPIYOY Index"},
        {"Country": "CHF", "Data": "CPI MoM", "Ticker": "SZCPIMOM Index"},

        {"Country": "CHF", "Data": "Unemployment Rate", "Ticker": "SZUE Index"},

        {"Country": "CHF", "Data": "GDP QoQ", "Ticker": "SZGDPCQQ Index"},
        {"Country": "CHF", "Data": "GDP YoY", "Ticker": "SZGRGDPY Index"},

        {"Country": "CHF", "Data": "SNB Policy Rate", "Ticker": "SZLTDEP Index"},
        {"Country": "CHF", "Data": "CB Foreign Reserves", "Ticker": "SZRAFCRC Index"},

        {"Country": "CHF", "Data": "Retail Sales YoY", "Ticker": "SZRSRYOY Index"},
        {"Country": "CHF", "Data": "PMI Manufacturing", "Ticker": "SZPUI Index"},
    ]

    manager = EconomicDataManager(bloomberg_api)
    manager.add_indicators('USD', USD_indicators)
    manager.add_indicators('EUR', EUR_indicators)
    manager.add_indicators('JPY', JPY_indicators)
    manager.add_indicators('GBP', GBP_indicators)
    manager.add_indicators('AUD', AUD_indicators)
    manager.add_indicators('NZD', NZD_indicators)
    manager.add_indicators('CHF', CHF_indicators)
    
    return manager

In [ ]:

# DataFrame: Cross Currency Vold Difference Z-Score --- (Z-Score)
def getCrossCCYVolSpread_ZScore(ccy_int, ccy_compare, tenor):
    def z_score(series, value):
        mu = series.mean()
        sigma = series.std(ddof=0)
        return np.nan if sigma == 0 else (value - mu) / sigma
    ccy_int = ccy_int[0] if isinstance(ccy_int, list) else ccy_int
    end_date = datetime.now()
    start_date = end_date - timedelta(days=365 * 5)  
    data_dict = {}
    for ccy in ccy_compare:
        ticker_IV = f"{ccy}V{tenor} BGN Curncy"
        data_IV = blp.bdh(
            tickers=ticker_IV,
            flds="PX_LAST", 
            start_date=start_date,
            end_date=end_date)
        data_IV.columns = data_IV.columns.get_level_values(1)
        data_dict[ccy] = data_IV['PX_LAST']
    df_all = pd.DataFrame(data_dict)
    df_all.index = pd.to_datetime(df_all.index)
    latest_date = df_all.index[-1]

    cut_3m  = latest_date - pd.DateOffset(months=3)
    cut_1y  = latest_date - pd.DateOffset(years=1)
    cut_3y  = latest_date - pd.DateOffset(years=3)
    cut_5y  = latest_date - pd.DateOffset(years=5)
    current_spreads_with_zscores = []

    for ccy in ccy_compare:
        if ccy != ccy_int:
            current_spread = df_all.loc[latest_date, ccy_int] - df_all.loc[latest_date, ccy]
            historical_spreads = df_all[ccy_int] - df_all[ccy]

            spreads_3m = historical_spreads.loc[historical_spreads.index >= cut_3m]
            spreads_1y = historical_spreads.loc[historical_spreads.index >= cut_1y]
            spreads_3y = historical_spreads.loc[historical_spreads.index >= cut_3y]
            spreads_5y = historical_spreads.loc[historical_spreads.index >= cut_5y]

            current_spreads_with_zscores.append({
                'Spread Pair'     : f"{ccy_int} - {ccy}",
                'Tenor'           : f"{tenor}",
                'Current Spread'  : round(current_spread, 4),
                '3M CrossCCY-Z'      : round(z_score(spreads_3m, current_spread), 2),
                '1Y CrossCCY-Z'      : round(z_score(spreads_1y, current_spread), 2),
                '3Y CrossCCY-Z'      : round(z_score(spreads_3y, current_spread), 2),
                '5Y CrossCCY-Z'      : round(z_score(spreads_5y, current_spread), 2),
            })

    ccys_spreads = pd.DataFrame(current_spreads_with_zscores).set_index("Tenor")
    return ccys_spreads[['Spread Pair', '3M CrossCCY-Z', '1Y CrossCCY-Z', '3Y CrossCCY-Z', '5Y CrossCCY-Z']]

In [12]:
df_next_releases = getReleaseDate_multiple(USD_indicators)
df_next_releases

NameError: name 'getReleaseDate_multiple' is not defined

In [11]:
ccy = ['USDBRL']
getRealImpDiff(ccy)

,1W R-I,2W R-I,3W R-I,1M R-I
Currency Pair,,,,
USDBRL,-4.0736,-3.42,-2.2268,-1.029


In [23]:
getVolAdjRR_ZScore(ccy)

,25D RR,3M RR-Z,1Y RR-Z,3Y RR-Z,5Y RR-Z
Tenor,,,,,
1W,2.9925,1.05,2.27,2.36,2.82
2W,3.0400,1.19,2.41,2.38,2.87
3W,3.0675,1.34,2.57,2.83,3.34
1M,3.0550,1.45,2.69,2.78,3.29


In [13]:
getVolAdjBF_ZScore(ccy)

,3M BF-Z,1Y BF-Z,3Y BF-Z,5Y BF-Z
Tenor,,,,
1W,-1.13,0.55,1.15,1.47
2W,-0.47,0.95,0.96,1.31
3W,-0.55,0.93,1.49,1.87
1M,-0.74,0.89,1.31,1.66


In [14]:
getTermSpread_ZScore(ccy)

,3M Term-Z,1Y Term-Z,3Y Term-Z,5Y Term-Z
Tenor Spread,,,,
1W 2W,1.94,0.64,0.39,0.38
1W 3W,2.12,0.71,0.53,0.49
1W 1M,2.27,0.76,0.62,0.59
2W 3W,1.48,0.45,0.46,0.44
2W 1M,1.83,0.59,0.60,0.59
3W 1M,1.92,0.76,0.77,0.77


In [141]:
ccy = ['EURUSD']
ccy_all = ['EURUSD', 'GBPUSD', 'USDJPY', 'AUDUSD', 'NZDUSD', 'USDCAD', 'USDCHF', 'USDNOK', 'USDSEK', 'USDMXN', 'USDBRL']
tenor = '1W'

crossCCYVols = getCrossCCYVolSpread_ZScore(ccy, ccy_all, tenor)

crossCCYVols

,Spread Pair,3M CrossCCY-Z,1Y CrossCCY-Z,3Y CrossCCY-Z,5Y CrossCCY-Z
Tenor,,,,,
1W,EURUSD - GBPUSD,0.44,0.60,1.05,1.07
1W,EURUSD - USDJPY,0.91,0.74,0.59,0.10
1W,EURUSD - AUDUSD,1.04,0.95,1.66,1.73
1W,EURUSD - NZDUSD,1.19,1.00,1.67,1.75
1W,EURUSD - USDCAD,-1.02,0.22,0.61,0.86
1W,EURUSD - USDCHF,1.11,0.26,0.10,0.02
1W,EURUSD - USDNOK,-0.87,0.44,1.14,1.45
1W,EURUSD - USDSEK,-0.10,0.45,1.09,0.88
1W,EURUSD - USDMXN,-0.34,0.85,0.72,0.94
